# 1. Preprocessing

In [178]:
import pandas as pd
import numpy as np

# static table
cities = pd.read_csv("../data/cities.csv")
stations = pd.read_csv(("../data/stations.csv"))
bikes = pd.read_csv(("../data/bikes.csv"))
bike_types = pd.read_csv(("../data/bike_types.csv"))

# dynamic table
trips = pd.read_csv(("../data/trips_small.csv"))

# Use external data from https://github.com/TUMFTM/european-bike-sharing-dataset/blob/main/sample/city_areas.csv
areas = pd.read_csv("../data/city_areas.csv")


## Util

**Solve this problem first: a bike has trips in multiple cities, but it should belong to one primary city**

Approach: If a bike has trips in multiple cities, it belongs to the city where it has the longest *total trip duration in that city*.

Solving this problem gives a usful dataframe **bike_primary_city** that can be reused in proceeding tasks.

In [179]:
print(f"number of cities in cities.csv: {cities["id"].nunique()}")
print(f"number of cities appeared in trips_small.csv: {trips["city_id"].nunique()}")

number of cities in cities.csv: 267
number of cities appeared in trips_small.csv: 185


In [180]:
print(f"Interesting discovery: bikes.csv has {bikes["id"].nunique()} bikes, but trips_small.csv has only trip records of {trips["bike_id"].nunique()} bikes.")

Interesting discovery: bikes.csv has 88436 bikes, but trips_small.csv has only trip records of 26424 bikes.


Interesting discovery: 
1. Among 267 cities only 185 cities have bike trip records.
2. bikes.csv has 88436 bikes, but trips_small.csv has only trip records of 26424 bikes.


In [181]:
# total trip durations of a bike in every city that it has trips.
tmp = (
    trips.groupby(["city_id", "bike_id"])["duration"]
    .sum().
    reset_index(name="total_duration")
)

# the useful dataframe that can be resued later
bike_primary_city = (tmp.sort_values(by="total_duration", ascending=False)
                     .drop_duplicates(subset="bike_id")
                     [["city_id", "bike_id"]]
                     )

print(f"{tmp.city_id.nunique()} cities has bike trips,")
print(f"but only {bike_primary_city.city_id.nunique()} primary cities are assigned,\nbecause there is one city where all bikes that appeared in that city had a higher total ride duration in other cities.")


185 cities has bike trips,
but only 184 primary cities are assigned,
because there is one city where all bikes that appeared in that city had a higher total ride duration in other cities.


In [182]:
cities_valid = bike_primary_city["city_id"].drop_duplicates()
print(f"number of valid cities: {cities_valid.shape[0]}")

number of valid cities: 184


We are only interested in these 184 cities, for we only have trip data in these cities. (The one non-primary city also excluded.)

## By organisational form: 149 station based, 35 free floating (including hybrid)

In [183]:

city_organization_form = (
    cities.merge(cities_valid, left_on="id", right_on="city_id", how="inner")[["id", "return_to_official_only"]]
    .rename(columns={"id": "city_id"})
)
city_organization_form["return_to_official_only"].value_counts()


return_to_official_only
True     149
False     35
Name: count, dtype: int64

## By propulsion type: 133 conventional, 38 mixed, 13 electric.

We also categorize BSSs by propulsion type: a city with 100% of human powered bikes is a conventional BSS, one with 0% human powered bikes is (fully) electric, while one in between is mixed.


In [184]:
# get its bike_types for every bike. Because bike_types specifies the propulsion type.
bikes = bikes.rename(columns={"id":"bike_id"})
bike_propulsion_type = (
    bikes.merge(bike_types, left_on="bike_type_id", right_on="id")
    .drop(columns=["id"])
    [["bike_id", "bike_type_id", "propulsion_type"]]
)

# check number of bikes with unidentified propsultion types
print(f"There are {bike_propulsion_type.shape[0]} bikes, {bike_propulsion_type["propulsion_type"].isna().sum()} bikes whose propulsion type cannot be identified.")
print("We drop those unidentified bikes, because they are few.")

# drop unidentified bikes
bike_propulsion_type = bike_propulsion_type.dropna(subset=["propulsion_type"])


There are 88436 bikes, 1050 bikes whose propulsion type cannot be identified.
We drop those unidentified bikes, because they are few.


Most of bikes are human powered:

In [185]:
bike_propulsion_type["propulsion_type"].value_counts()


propulsion_type
human              81530
electric_assist     5856
Name: count, dtype: int64

In [186]:
# merge with bike_primary_city
city_bike_propulsion = bike_propulsion_type.merge(bike_primary_city, on="bike_id", how="inner")
city_bike_propulsion.shape

(25833, 4)

Note that all bikes from trips has been assigned with their (primary) cities. But some bikes have NA in propulsion type--those bikes are dropped in bikes_propulsion_type. By inner merge, these bikes are dropped. The resulting city_bike_propulsion has cities with all bikes whose propulsion types are known.
(25833 < 26424)

Final step: calculate the proportion of human-powered bikes and categorize cities by propulsion types into conventional, mixed, electric.

In [187]:
# count the proportion of human powered bikes in each city
tmp = (city_bike_propulsion.assign(is_human=(city_bike_propulsion["propulsion_type"] == "human"))
        .groupby("city_id")["is_human"]
        .mean()
        .reset_index(name="proportion_of_human_powered_bikes")
        )

# define filter
def categorize_propulsion_type(n):
    if n == 1:
        return "conventional"
    elif n == 0:
        return "electric"
    else:
        return "mixed"

# apply filter
tmp["type"] = tmp["proportion_of_human_powered_bikes"].apply(categorize_propulsion_type)

city_propulsion_type = tmp
city_propulsion_type["type"].value_counts()

type
conventional    133
mixed            38
electric         13
Name: count, dtype: int64

## n_bikes in each city

**city_bikes** city_id | n_bikes

In [188]:
city_bikes = (city_bike_propulsion.groupby("city_id")["bike_id"]
              .count()
              .reset_index(name="n_bikes"))


## thus Categorizing BSS by size
Small systems: 10 <= n_bikes <= 100
Medium: 101 <= n_bikes <= 500
Large: 501 <= n_bikes

In [189]:
# define filter
def categorize_by_size(n):
    if n >= 500:
        return "large"
    elif n >= 101:
        return "medium"
    elif n >= 10:
        return "small"
    return None

# apply filter
city_size = city_bikes
city_size["size"] = city_size["n_bikes"].apply(categorize_by_size)

print(f"number of small BSS: {city_size[city_size["size"] == "small"].shape[0]}")
print(f"number of medium BSS: {city_size[city_size["size"] == "medium"].shape[0]}")
print(f"number of large BSS: {city_size[city_size["size"] == "large"].shape[0]}")
print(f"{city_size["size"].isna().sum()} BSSs are too small (<10 bikes), they should be excluded from the analysis.")


number of small BSS: 97
number of medium BSS: 72
number of large BSS: 9
6 BSSs are too small (<10 bikes), they should be excluded from the analysis.


In [190]:
# drop too small BSSs
city_size = city_size.dropna(subset="size")
city_size.shape

(178, 3)

## n_stations in each city

**city_stations:** city_id | n_stations

In [191]:
stations = pd.read_csv("../data/stations.csv")

city_stations = (
    stations.groupby("city_id")["id"]
    .count()
    .reset_index(name="n_stations")
)

In [192]:
city_stations.head()

,city_id,n_stations
0,7,37
1,23,29
2,64,44
3,88,41
4,128,44


## n_racks in each city

We noticed that some special racks has negative values. Using AI to search info: there are two reporting schemes from Nextbike. If special_racak is positive, bike_racks means regular racks; if negative, bike_racks means number of total racks, the "-" sign means reserved for charging e-bikes. 

In [193]:
stations["total_racks"] = np.where(
    stations["special_racks"] < 0,
    stations["bike_racks"],                                      # subtractive: bike_racks is already gross total
    stations["bike_racks"] + stations["special_racks"].fillna(0) # additive: add both
)

# city_bikes we already have, now calculate city_racks
city_racks = (
    stations.groupby("city_id")["total_racks"]
    .sum()
    .reset_index(name="n_racks")
)


## city area

use city_areas.csv from github, meaning cf paper Table 13

**city_area**: city_id 

In [194]:
# get the city area (operational area) in sqare km.

# between two lines of "===" are AI generated code
# ====================================================================================
import pandas as pd
import geopandas as gpd
from shapely import wkt

# Parse geometries from WKT (stripping the "SRID=4326;" prefix)
areas['geometry'] = areas['geom_ewkt'].apply(lambda x: wkt.loads(x.split(';')[-1]))

# Create GeoDataFrame in WGS84
gdf = gpd.GeoDataFrame(areas, geometry='geometry', crs="EPSG:4326")

# Option A: Geodesic area calculation directly in km²
# gdf.to_crs(epsg=6933) is an Equal-Area projection (meters)
gdf_projected = gdf.to_crs(epsg=6933)
gdf['area_sq_km'] = gdf_projected.geometry.area / 1_000_000
# ====================================================================================

city_areas = gdf[["city_id", "area_sq_km"]]


## FINAL df

**df**: city_id | size | n_stations| n_bikes| n_racks | area | organization type | propulsion type

Merge all

In [195]:
###
df = (city_size
      .merge(city_stations, on="city_id")
      .merge(city_organization_form, on="city_id")
      .merge(city_propulsion_type, on="city_id")
      .merge(city_racks, on="city_id")
      .merge(city_areas, on="city_id")
    )

df.head()

,city_id,n_bikes,size,n_stations,return_to_official_only,proportion_of_human_powered_bikes,type,n_racks,area_sq_km
0,7,456,medium,37,False,1.0,conventional,24,23.608454
1,23,97,small,29,True,1.0,conventional,0,483.600722
2,64,340,medium,44,True,1.0,conventional,0,61.150020
3,88,266,medium,41,True,1.0,conventional,0,37.598203
4,128,235,medium,44,True,1.0,conventional,251,11.922279


# 2. Descriptive Analysis